<a href="https://colab.research.google.com/github/himanshusar123/-Machine-Learning-Quiz-Classification-or-Regression-/blob/main/Day_3_updated_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import csv
import os
import sqlite3

# Configuration
DB_NAME = "securebank.db"
CSV_NAME = "Day3_SecureBank_Customer_Master.csv"


def get_db_connection():
    """Establishes and returns a connection to the SQLite database."""
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row  # Enables accessing columns by name
    return conn


# ==========================================
# SPRINT 1: Database Setup
# ==========================================
def create_database():
    """Creates the SQLite database and the Customers table with appropriate schema."""
    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        cursor.execute(
            """
            CREATE TABLE IF NOT EXISTS Customers (
                CustomerID TEXT PRIMARY KEY,
                CustomerName TEXT NOT NULL,
                Branch TEXT NOT NULL,
                AccountType TEXT NOT NULL,
                Balance REAL NOT NULL,
                Status TEXT NOT NULL DEFAULT 'Active'
            )
        """
        )
        conn.commit()
        print(
            f"\n[SUCCESS] Database '{DB_NAME}' and 'Customers' table are ready."
        )
    except sqlite3.Error as e:
        print(f"\n[ERROR] Failed to create database: {e}")
    finally:
        conn.close()


# ==========================================
# SPRINT 2: CSV Migration
# ==========================================
def import_csv():
    """Imports unique records from the provided CSV dataset into the SQLite database."""
    if not os.path.exists(CSV_NAME):
        print(
            f"\n[ERROR] Source data file '{CSV_NAME}' not found in the current directory."
        )
        return

    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        with open(CSV_NAME, mode="r", encoding="utf-8") as file:
            csv_reader = csv.DictReader(file)

            # Strip whitespace from header field names to prevent mapping errors
            csv_reader.fieldnames = [
                field.strip() for field in csv_reader.fieldnames
            ]

            inserted_count = 0
            skipped_count = 0

            for row in csv_reader:
                customer_id = row["Customer ID"].strip()

                # Validate if the Customer ID already exists to respect PRIMARY KEY integrity
                cursor.execute(
                    "SELECT 1 FROM Customers WHERE CustomerID = ?",
                    (customer_id,),
                )
                if cursor.fetchone():
                    skipped_count += 1
                    continue

                cursor.execute(
                    """
                    INSERT INTO Customers (CustomerID, CustomerName, Branch, AccountType, Balance, Status)
                    VALUES (?, ?, ?, ?, ?, ?)
                """,
                    (
                        customer_id,
                        row["Customer Name"].strip(),
                        row["Branch"].strip(),
                        row["Account Type"].strip(),
                        float(row["Balance"]),
                        row["Status"].strip(),
                    ),
                )
                inserted_count += 1

        conn.commit()
        print(f"\n[MIGRATION COMPLETE]")
        print(f"-> Records successfully imported: {inserted_count}")
        print(f"-> Records skipped (Duplicates encountered): {skipped_count}")

    except Exception as e:
        conn.rollback()
        print(f"\n[ERROR] Data migration failed: {e}")
    finally:
        conn.close()


# ==========================================
# SPRINT 3: Display Customers
# ==========================================
def display_customers():
    """Retrieves and lists all records inside the database table."""
    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        cursor.execute("SELECT * FROM Customers")
        rows = cursor.fetchall()

        if not rows:
            print("\n[INFO] The Customers database is currently empty.")
            return

        print(
            f"\n{'ID':<8} | {'Customer Name':<20} | {'Branch':<15} | {'Type':<10} | {'Balance':<12} | {'Status':<10}"
        )
        print("-" * 80)
        for row in rows:
            print(
                f"{row['CustomerID']:<8} | {row['CustomerName']:<20} | {row['Branch']:<15} | {row['AccountType']:<10} | {row['Balance']:<12,.2f} | {row['Status']:<10}"
            )

    except sqlite3.Error as e:
        print(f"\n[ERROR] Could not fetch customer records: {e}")
    finally:
        conn.close()


# ==========================================
# SPRINT 4: Search Customer
# ==========================================
def search_customer():
    """Searches a specific customer record via parameterized Customer ID."""
    cust_id = input("\nEnter Customer ID to search: ").strip()

    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        cursor.execute(
            "SELECT * FROM Customers WHERE CustomerID = ?", (cust_id,)
        )
        row = cursor.fetchone()

        if row:
            print(f"\n--- Customer Profile: {cust_id} ---")
            print(f"Name:         {row['CustomerName']}")
            print(f"Branch:       {row['Branch']}")
            print(f"Account Type: {row['AccountType']}")
            print(f"Balance:      INR {row['Balance']:,.2f}")
            print(f"Status:       {row['Status']}")
        else:
            print(f"\n[NOT FOUND] No customer record matches ID: {cust_id}")

    except sqlite3.Error as e:
        print(f"\n[ERROR] Problem querying record: {e}")
    finally:
        conn.close()


# ==========================================
# SPRINT 5: Add Customer
# ==========================================
def add_customer():
    """Validates parameters and inserts a unique record into the database."""
    print("\n--- Add New Customer Profile ---")
    cust_id = input("Enter Customer ID (e.g., C1011): ").strip()

    if not cust_id:
        print("[INVALID] Customer ID cannot be left blank.")
        return

    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        # Step 1: Validate Duplicate Primary Keys
        cursor.execute(
            "SELECT 1 FROM Customers WHERE CustomerID = ?", (cust_id,)
        )
        if cursor.fetchone():
            print(
                f"[REJECTED] Customer ID {cust_id} already exists in the system."
            )
            return

        name = input("Enter Customer Name: ").strip()
        branch = input("Enter Branch: ").strip()
        acc_type = input("Enter Account Type (Savings/Current/Salary): ").strip()

        try:
            balance = float(input("Enter Initial Balance: "))
            if balance < 0:
                print("[REJECTED] Initial balance cannot be negative.")
                return
        except ValueError:
            print("[INVALID] Balance must be a valid numeric value.")
            return

        status = "Active"  # Default status for structural integrity

        # Step 2: Insert data using parameterized mapping
        cursor.execute(
            """
            INSERT INTO Customers (CustomerID, CustomerName, Branch, AccountType, Balance, Status)
            VALUES (?, ?, ?, ?, ?, ?)
        """,
            (cust_id, name, branch, acc_type, balance, status),
        )

        conn.commit()
        print(
            f"\n[SUCCESS] Profile created successfully for {name} ({cust_id})."
        )

    except sqlite3.Error as e:
        conn.rollback()
        print(f"\n[ERROR] System failed to save record: {e}")
    finally:
        conn.close()


# ==========================================
# SPRINT 6: Update Customer
# ==========================================
def update_customer():
    """Modifies customer configuration rules conditionally dynamically."""
    cust_id = input("\nEnter Customer ID to modify: ").strip()

    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        cursor.execute(
            "SELECT * FROM Customers WHERE CustomerID = ?", (cust_id,)
        )
        row = cursor.fetchone()

        if not row:
            print(f"[NOT FOUND] Customer profile {cust_id} does not exist.")
            return

        print(f"\nModifying attributes for {row['CustomerName']} ({cust_id})")
        print("Leave field blank and press Enter to keep current system setting.")

        new_branch = (
            input(f"Branch [{row['Branch']}]: ").strip() or row['Branch']
        )
        new_acc_type = (
            input(f"Account Type [{row['AccountType']}]: ").strip()
            or row['AccountType']
        )

        current_balance = row["Balance"]
        balance_input = input(f"Balance [{current_balance}]: ").strip()
        if balance_input:
            try:
                new_balance = float(balance_input)
                if new_balance < 0:
                    print("[INVALID] Balance cannot be negative. Keeping old value.")
                    new_balance = current_balance
            except ValueError:
                print("[INVALID] Non-numeric value input. Keeping old value.")
                new_balance = current_balance
        else:
            new_balance = current_balance

        new_status = (
            input(f"Status [{row['Status']}]: ").strip() or row['Status']
        )

        cursor.execute(
            """
            UPDATE Customers
            SET Branch = ?, AccountType = ?, Balance = ?, Status = ?
            WHERE CustomerID = ?
        """,
            (new_branch, new_acc_type, new_balance, new_status, cust_id),
        )

        conn.commit()
        print(f"\n[SUCCESS] Customer profile {cust_id} has been modified.")

    except sqlite3.Error as e:
        conn.rollback()
        print(f"\n[ERROR] Update transaction halted: {e}")
    finally:
        conn.close()


# ==========================================
# SPRINT 7: Deactivate Customer
# ==========================================
def deactivate_customer():
    """Soft deletes customer metadata records through boolean/status flags."""
    cust_id = input("\nEnter Customer ID to deactivate: ").strip()